# Create Shriners Hospitals for Children Awards

Builds the awards table for **Shriners Hospitals for Children** (funder_id
`4320313002`, IAMHRF expanded list, priority `287`) from their public
ProposalCentral Insights award search (GMID=83). ~93 currently-funded grants
with native `grantID`, project title, PI (+ ORCID where available), USD amount,
start/end dates, and the funding institution (a Shriners hospital site).

Source parquet: `s3://openalex-ingest/awards/shriners/shriners_grants.parquet`
(produced by `scripts/local/shriners_to_s3.py`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.shriners_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/shriners/shriners_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be ~93)
SELECT COUNT(*) as total_projects FROM openalex.awards.shriners_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.shriners_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.shriners_raw LIMIT 5;

## Step 2: Create Shriners Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.shriners_awards
USING delta
AS
WITH
shriners_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320313002  -- Shriners Hospitals for Children
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'USD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.program as funder_scheme,
        'shriners' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'MM/dd/yyyy') as start_date,
        TRY_TO_DATE(g.end_date_raw, 'MM/dd/yyyy') as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'MM/dd/yyyy')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'MM/dd/yyyy')) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    g.orcid as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        CASE
                            WHEN g.institution ILIKE '%Canada%' THEN 'Canada'
                            WHEN g.institution ILIKE '%Mexico%' THEN 'Mexico'
                            ELSE 'United States'
                        END as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.shriners_raw g
    CROSS JOIN shriners_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'shriners' AND priority = 287;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    287 as priority
FROM openalex.awards.shriners_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count (should be ~93)
SELECT COUNT(*) as total_awards FROM openalex.awards.shriners_awards;

In [ ]:
%sql
-- Sample the data
SELECT
    id, display_name, funder_award_id, funder_scheme,
    funding_type, amount, currency, start_year, end_year
FROM openalex.awards.shriners_awards
LIMIT 10;

In [ ]:
%sql
-- Check funder distribution (should all be Shriners Hospitals for Children)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.shriners_awards
GROUP BY funder.display_name
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check program/scheme distribution
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount)/1000000, 2) as funding_million_usd
FROM openalex.awards.shriners_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Check data completeness
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.orcid) as has_orcid,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_with_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million_usd
FROM openalex.awards.shriners_awards;

In [ ]:
%sql
-- Check top institutions (Shriners hospital sites)
SELECT lead_investigator.affiliation.name as institution,
       lead_investigator.affiliation.country as country,
       COUNT(*) as cnt
FROM openalex.awards.shriners_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1, 2
ORDER BY cnt DESC
LIMIT 20;

In [ ]:
%sql
-- Verify data was inserted into openalex_awards_raw
SELECT COUNT(*) as shriners_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'shriners' AND priority = 287;